# Dynamic Topic Modeling for Cluster 6: Work, Jobs & Workplace Life

This notebook uses BERTopic to explore potential topics within **Cluster 6** (Work, jobs, workplace life & worker communities) from the subreddit clustering analysis.

**Pipeline:**
1. Load data and filter to cluster 6 subreddits
2. Preprocess text (handle noise words via CountVectorizer stopwords)
3. Fit BERTopic with KeyBERTInspired representation (reduces noise words)
4. Run dynamic topic modeling (topics over time)
5. Visualize results

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

import warnings
warnings.filterwarnings('ignore')

## 2. Load Data & Filter Cluster 6

In [ ]:
# ===== CONFIGURATION =====
# Change this path to your full dataset when ready
DATA_PATH = "../reddit_cleaned_01_13_first10.csv"
CLUSTER_PATH = "../subreddit_cluster_summary_k20.csv"
TARGET_CLUSTER = 6
TEXT_COLUMN = "merged_text"
TIME_COLUMN = "created_utc"
# =========================

In [ ]:
# Load cluster summary and extract cluster 6 subreddits
cluster_df = pd.read_csv(CLUSTER_PATH)
c6 = cluster_df[cluster_df["cluster"] == TARGET_CLUSTER]

print(f"Cluster {TARGET_CLUSTER}: {c6['Name by LLM'].values[0]}")
print(f"Top words: {c6['top_words'].values[0]}")
print(f"Expected subreddits: {c6['n_subreddits'].values[0]}, expected rows: {c6['n_rows'].values[0]}")

# Parse subreddits (semicolon-separated)
cluster6_subreddits = [s.strip() for s in c6["subreddits"].values[0].split(";")]
print(f"\nSubreddits in cluster 6 ({len(cluster6_subreddits)}):")
print(cluster6_subreddits)

In [ ]:
# Load the full dataset
df_all = pd.read_csv(DATA_PATH)
print(f"Total rows in dataset: {len(df_all)}")

# Filter to cluster 6 subreddits only
df = df_all[df_all["subreddit"].isin(cluster6_subreddits)].copy()
print(f"Rows matching cluster 6 subreddits: {len(df)}")

if len(df) < 20:
    print(f"\n⚠️  WARNING: Only {len(df)} documents found for cluster 6.")
    print("   BERTopic needs more documents for meaningful topic discovery.")
    print("   Please update DATA_PATH to your full dataset CSV.")
    print("   Continuing with available data for demonstration...")

print(f"\nSubreddits found: {df['subreddit'].unique().tolist()}")
print(f"Rows with {TEXT_COLUMN}: {df[TEXT_COLUMN].notna().sum()}")

In [ ]:
# Drop rows with missing text and prepare documents
df = df.dropna(subset=[TEXT_COLUMN]).reset_index(drop=True)

# Parse timestamps
df[TIME_COLUMN] = pd.to_datetime(df[TIME_COLUMN])

docs = df[TEXT_COLUMN].tolist()
timestamps = df[TIME_COLUMN].tolist()

print(f"Documents ready: {len(docs)}")
print(f"Time range: {df[TIME_COLUMN].min()} to {df[TIME_COLUMN].max()}")
print(f"\nSample doc (first 300 chars):")
print(docs[0][:300])

## 3. Configure BERTopic with Noise Reduction

Key strategies to reduce noise words ("sisters", "that", "like", etc.):
1. **CountVectorizer** with `stop_words="english"` removes common English stopwords
2. **Custom stopwords** for domain-specific noise (e.g., Reddit jargon)
3. **KeyBERTInspired** representation model produces more coherent, less noisy topic labels
4. **min_df** filters out very rare terms; **ngram_range** captures multi-word phrases

In [ ]:
# Custom stopwords: add domain-specific noise words here
# These are words that appear frequently but don't help distinguish topics
custom_stopwords = [
    # Reddit-specific noise
    "like", "just", "got", "get", "going", "would", "could", "really",
    "also", "know", "think", "want", "even", "still", "much", "thing",
    "things", "way", "make", "made", "said", "one", "people", "time",
    "don", "didn", "doesn", "isn", "wasn", "won", "wouldn", "couldn",
    "shouldn", "hasn", "hadn", "aren", "weren", "ll", "ve",
    # Conversational noise commonly seen in Reddit posts/comments
    "yeah", "okay", "lol", "lmao", "edit", "update", "deleted",
    "comment", "comments", "post", "posted", "thread", "subreddit",
    "reddit", "op", "username",
    # Relationship/story noise often appearing in work-related posts
    "sister", "sisters", "brother", "husband", "wife", "friend",
    "mom", "dad", "family",
]

# Merge with sklearn's English stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
all_stopwords = list(ENGLISH_STOP_WORDS.union(custom_stopwords))

print(f"Total stopwords: {len(all_stopwords)}")

In [ ]:
# --- Sub-models ---

# Embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# UMAP for dimensionality reduction
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

# HDBSCAN for clustering
hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

# CountVectorizer with stopwords and ngrams
vectorizer_model = CountVectorizer(
    stop_words=all_stopwords,
    min_df=2,
    ngram_range=(1, 2),  # unigrams + bigrams for richer topic labels
)

# KeyBERTInspired for cleaner topic representations
representation_model = KeyBERTInspired()

print("Sub-models configured.")

## 4. Fit BERTopic Model

In [ ]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    top_n_words=10,
    verbose=True,
)

topics, probs = topic_model.fit_transform(docs)

print(f"\nNumber of topics found: {len(set(topics)) - (1 if -1 in topics else 0)}")
print(f"Outlier documents (topic -1): {topics.count(-1)}/{len(topics)}")

In [ ]:
# View discovered topics
topic_info = topic_model.get_topic_info()
print("Discovered topics:")
topic_info

In [ ]:
# Show top words for each topic
for topic_id in topic_info[topic_info["Topic"] != -1]["Topic"]:
    words = [w for w, _ in topic_model.get_topic(topic_id)]
    print(f"Topic {topic_id}: {', '.join(words[:8])}")

## 5. Static Visualizations

In [ ]:
# Topic word barchart
fig = topic_model.visualize_barchart(top_n_topics=10, n_words=8)
fig.show()

In [ ]:
# Intertopic distance map
fig = topic_model.visualize_topics()
fig.show()

In [ ]:
# Topic hierarchy
fig = topic_model.visualize_hierarchy()
fig.show()

## 6. Dynamic Topic Modeling (Topics Over Time)

This calculates how topic representations evolve across timestamps.

**Note:** Keep unique timestamps below ~50 for best results. Use `nr_bins` to bin timestamps.

In [ ]:
# Check unique timestamps
unique_dates = df[TIME_COLUMN].dt.date.nunique()
print(f"Unique dates: {unique_dates}")

# Choose nr_bins: keep it manageable
nr_bins = min(20, unique_dates)
print(f"Using nr_bins={nr_bins}")

In [ ]:
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps,
    nr_bins=nr_bins,
    evolution_tuning=True,
    global_tuning=True,
)

print(f"Topics over time shape: {topics_over_time.shape}")
topics_over_time.head(10)

In [ ]:
# Visualize topics over time
fig = topic_model.visualize_topics_over_time(
    topics_over_time,
    top_n_topics=10,
)
fig.show()

## 7. Topics Per Subreddit (Optional Analysis)

In [ ]:
# Analyze which topics appear in which subreddits
subreddits = df["subreddit"].tolist()
topics_per_class = topic_model.topics_per_class(docs, classes=subreddits)

fig = topic_model.visualize_topics_per_class(
    topics_per_class,
    top_n_topics=10,
)
fig.show()

## 8. Explore Specific Topics

In [ ]:
# Find representative documents for each topic
for topic_id in sorted(set(topics)):
    if topic_id == -1:
        continue
    repr_docs = topic_model.get_representative_docs(topic_id)
    words = [w for w, _ in topic_model.get_topic(topic_id)]
    print(f"\n{'='*80}")
    print(f"Topic {topic_id}: {', '.join(words[:6])}")
    print(f"{'='*80}")
    for i, doc in enumerate(repr_docs[:2]):
        print(f"  Doc {i+1}: {doc[:200]}...")

## 9. Save Results

In [ ]:
# Save topic assignments back to the dataframe
df["bertopic_topic"] = topics
df["bertopic_prob"] = [p.max() if hasattr(p, 'max') else p for p in probs]

# Save to CSV
output_path = "../cluster6_topic_results.csv"
df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")

# Save topics over time
tot_path = "../cluster6_topics_over_time.csv"
topics_over_time.to_csv(tot_path, index=False)
print(f"Topics over time saved to {tot_path}")

In [ ]:
# Save the model (optional - uncomment to save)
# topic_model.save("../cluster6_bertopic_model", serialization="safetensors", save_ctfidf=True)

---

## Tips for Better Results

**If you still see noise words in topics:**
1. Add them to `custom_stopwords` in Section 3 and re-run from there
2. Increase `min_df` in the CountVectorizer (e.g., `min_df=5`)
3. Try `MaximalMarginalRelevance` as an alternative representation model:
   ```python
   from bertopic.representation import MaximalMarginalRelevance
   representation_model = MaximalMarginalRelevance(diversity=0.3)
   ```

**If too many documents are outliers (topic -1):**
1. Lower `min_cluster_size` in HDBSCAN (e.g., `min_cluster_size=10`)
2. Use `topic_model.reduce_outliers(docs, topics)` to reassign outliers

**If too few / too many topics:**
1. Set `nr_topics` in BERTopic constructor to control the number (e.g., `nr_topics=10`)
2. Or use `nr_topics="auto"` for automatic reduction
3. Adjust `min_cluster_size` — larger values = fewer, broader topics